Clean Classifer Script for reproduction of results

load data and dependencies

In [ ]:
import pandas as pd
import sys
import os

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)
import pickle
import spacy
from scipy.stats import pearsonr, spearmanr, ttest_ind


os.chdir(r"YOUR_WORKING_DIRECTORY")
sys.path.append(os.path.abspath(r"YOUR_WORKING_DIRECTORY"))



from src.functions import bootstrap_ci
from src.classifier_function import (
    make_pipelines,
    run_nested_cv,
    classifier_builder_multi,
    make_pipelines_ngram,
    compare_best_families,
    pick_best_family,
    permutation_test_nested_cv,
    run_symptom_cluster_classifiers,
    run_cutoff_sensitivity,
    particiapnt_level_single_ci,
    report_precision_recall,
    run_dass_subscale_classifiers,
    report_best_family_metrics,
    make_pipelines_ngram_fixed
)




dataset = pd.read_csv("data/dataset_anonymized_for_release.csv", sep=";", decimal=".", encoding="utf-8")
textvariables = [
    "response_negnt",
    "response_neutr",
    "response_trauma"
]



#============================================================
1. NARRATIVE CONDITIONS: trauma / negative / neutral
(single run — reused everywhere below, never rerun)

#============================================================

In [8]:
target_col = "condition_pcl" #column that holds the information about a participants PTSD group alignment
positive_class = "clin_pcl" #positive class tag (high PTSD)

In [11]:
condition_results, complete_dataset, embedding_cols_by_condition, y_data, pipelines_list = classifier_builder_multi(
    dataset=dataset,
    target_col=target_col,
    positive_class=positive_class,
    response_types=("response_trauma", "response_negnt", "response_neutr"),
) #function that builds and runs a pipeline for a nested cross validation run including the run_nested_cv function
#that is done here so direct matching of folds is possible for comparisons
#for more details see "src.classifier_functions"

Complete-case N (valid embeddings across all conditions): 150
Positive class proportion: 0.2733333333333333

CONDITION: response_trauma

--- Outer Fold 1 ---
Overall best inner: ('gbt', 'depth2', 200), inner AUC=0.694
Overall outer test: AUC=0.710, Acc=0.767
  [gbt] best inner cfg=('depth2', 200), outer AUC=0.710
  [logreg] best inner cfg=('class_weight', 0.01), outer AUC=0.767
  [svm] best inner cfg=('no_weight', 0.01), outer AUC=0.818

--- Outer Fold 2 ---
Overall best inner: ('gbt', 'depth2', 100), inner AUC=0.759
Overall outer test: AUC=0.494, Acc=0.700
  [gbt] best inner cfg=('depth2', 100), outer AUC=0.494
  [logreg] best inner cfg=('no_weight', 100.0), outer AUC=0.670
  [svm] best inner cfg=('no_weight', 0.01), outer AUC=0.670

--- Outer Fold 3 ---
Overall best inner: ('logreg', 'class_weight', 100.0), inner AUC=0.681
Overall outer test: AUC=0.716, Acc=0.733
  [gbt] best inner cfg=('depth2', 100), outer AUC=0.670
  [logreg] best inner cfg=('class_weight', 100.0), outer AUC=0.716

KeyboardInterrupt: 

1b. VERIFY: outer fold membership is identical across conditions, which allows the participant-level paired comparisons in 1c

In [ ]:
trauma_folds = list(condition_results["response_trauma"]["outer_cv"].split(
    np.zeros(len(y_data)), y_data
))
negnt_folds = list(condition_results["response_negnt"]["outer_cv"].split(
    np.zeros(len(y_data)), y_data
))
neutr_folds = list(condition_results["response_neutr"]["outer_cv"].split(
    np.zeros(len(y_data)), y_data
))

fold_check_passed = True
for i in range(5):
    match_negnt = np.array_equal(trauma_folds[i][1], negnt_folds[i][1])
    match_neutr = np.array_equal(trauma_folds[i][1], neutr_folds[i][1])
    print(f"Fold {i}: trauma vs negnt = {match_negnt}, trauma vs neutr = {match_neutr}")
    fold_check_passed &= match_negnt and match_neutr

assert fold_check_passed, "Outer fold membership differs across conditions — participant-level pairing is invalid."
print("\nConfirmed: outer fold membership is identical across all three narrative conditions.")

1c. Pairwise Comparisons (participant-level paired bootstrap)

In [ ]:
def compare(label_a, label_b):
    pa = all_analyses[label_a]["per_family_pooled"][final_metrics[label_a]["family"]]
    pb = all_analyses[label_b]["per_family_pooled"][final_metrics[label_b]["family"]]
    assert np.array_equal(pa["participant_idx"], pb["participant_idx"])
    diff, lo, hi, _ = participant_level_paired_bootstrap(pa["y_true"], pa["y_score"], pb["y_score"])
    print(f"{label_a} vs {label_b}: diff={diff:.3f}, 95% CI=[{lo:.3f}, {hi:.3f}]")
    return diff, lo, hi

pairwise_comparisons = {
    "Trauma_vs_Negative": compare("Trauma", "Negative"),
    "Trauma_vs_Neutral": compare("Trauma", "Neutral"),
    "Trauma_vs_Ngram": compare("Trauma", "N-gram baseline"),
    "Trauma_vs_Subjective": compare("Trauma", "Subjective measures"),
}

#============================================================

2. N-GRAM BASELINE (3-family grid, matching embedding comparison)

#============================================================

In [ ]:
X_text_trauma = complete_dataset["response_trauma"].astype(str).to_numpy()

ngram_results = run_nested_cv(
    X_text_trauma, y_data, make_pipelines_ngram_fixed(),
    n_outer_splits=5, n_inner_splits=5,
) #uses the actual engine of the nested cv, without the classifier_builder_multi wrapper

#============================================================

3. SUBJECTIVE-MEASURES CLASSIFIER

#============================================================

In [ ]:
subjective_feature_cols = [
    f"{p}_trauma_1" for p in
    ["vivid", "emoArous", "valence", "detail", "relevance", "imagery", "perspec", "active", "ease"]
] #columns that contain a participants subjective detail on the trauma narratives
sub_dataset = dataset.dropna(subset=subjective_feature_cols + [target_col]).reset_index(drop=True)
X_subjective = sub_dataset[subjective_feature_cols].to_numpy()
y_subjective = ((sub_dataset[target_col] == positive_class) * 1).to_numpy()

subjective_results = run_nested_cv(X_subjective, y_subjective, make_pipelines(), n_outer_splits=5, n_inner_splits=5)
#same as above


#============================================================

4. PCL-5 / DASS-21 CORRELATIONS

#============================================================

In [ ]:
pcl5_col = "PCL-5_total" #numerical value of the combined PCL-5 items
dass_cols = {
    "DASS-21 total": "dass21_total",
    "DASS-21 depression": "DASS21_depression",
    "DASS-21 anxiety": "DASS21_anxiety",
    "DASS-21 stress": "DASS21_stress",
} #numerical value of the DASS-21 subscales
correlations_spearman = {}
for label, col in dass_cols.items():
    sub = dataset[[pcl5_col, col]].dropna()
    r, p = spearmanr(sub[pcl5_col], sub[col])
    print(f"{pcl5_col} vs {col}: rho={r:.3f}, p={p:.4f}, n={len(sub)}")
    correlations_spearman[label] = (r, p) #correlation test for the DASS-21 subscales and the PCL-5 score

#============================================================

5. DASS-21 SUBSCALE CLASSIFIERS

#============================================================

In [ ]:
dass_targets = {
    "DASS21_depression": ("condition_dass_depression", "clin_dass_depression"),
    "DASS21_anxiety": ("condition_dass_anxiety", "clin_dass_anxiety"),
    "DASS21_stress": ("condition_dass_stress", "clin_dass_stress"),
}
embedding_cols_trauma = embedding_cols_by_condition["response_trauma"]

dass_results = {}
for subscale, (tcol, pclass) in dass_targets.items(): #wrapper to run the nested_cv for each of the DASS-21 subscales
    sub_ds = dataset.dropna(subset=embedding_cols_trauma + [tcol]).reset_index(drop=True)
    X = sub_ds[embedding_cols_trauma].to_numpy()
    y = ((sub_ds[tcol] == pclass) * 1).to_numpy()
    dass_results[subscale] = run_nested_cv(X, y, make_pipelines(), n_outer_splits=5, n_inner_splits=5)#nested cv run


#============================================================

6. DSM-5 SYMPTOM CLUSTER CLASSIFIERS

#============================================================

In [ ]:
cluster_items = { #creating the DSM-5 clusters from the PCL-5 items representing the cluster
    "Intrusion": [f"PCL-5_{i}" for i in range(1, 6)],
    "Avoidance": [f"PCL-5_{i}" for i in range(6, 8)],
    "NegCognitionsMood": [f"PCL-5_{i}" for i in range(8, 15)],
    "ArousalReactivity": [f"PCL-5_{i}" for i in range(15, 21)],
}
for cluster_name, items in cluster_items.items(): #do a median split on these items to categorize people into two groups
    dataset[f"PCL5_{cluster_name}"] = dataset[items].sum(axis=1)
    median_val = dataset[f"PCL5_{cluster_name}"].median()
    dataset.loc[dataset[f"PCL5_{cluster_name}"] > median_val, f"condition_{cluster_name}"] = f"high_{cluster_name}"
    dataset.loc[dataset[f"PCL5_{cluster_name}"] <= median_val, f"condition_{cluster_name}"] = f"low_{cluster_name}"

cluster_targets = { #make the cluster targets so again, a wrapper can be used for the nested_cv_runs
    "Intrusion": ("condition_Intrusion", "high_Intrusion"),
    "Avoidance": ("condition_Avoidance", "high_Avoidance"),
    "NegCognitionsMood": ("condition_NegCognitionsMood", "high_NegCognitionsMood"),
    "ArousalReactivity": ("condition_ArousalReactivity", "high_ArousalReactivity"),
}
cluster_results = {}
for cluster, (tcol, pclass) in cluster_targets.items():
    sub_ds = dataset.dropna(subset=embedding_cols_trauma + [tcol]).reset_index(drop=True)
    X = sub_ds[embedding_cols_trauma].to_numpy()
    y = ((sub_ds[tcol] == pclass) * 1).to_numpy()
    cluster_results[cluster] = run_nested_cv(X, y, make_pipelines(), n_outer_splits=5, n_inner_splits=5)


#============================================================

7. PCL-5 CUTOFF SENSITIVITY

#============================================================

In [ ]:
cutoff_results = {}
for cutoff in (31, 33, 34, 36, 38):
    sub_ds = dataset.dropna(subset=embedding_cols_trauma + [pcl5_col]).reset_index(drop=True)
    X = sub_ds[embedding_cols_trauma].to_numpy()
    y = (sub_ds[pcl5_col] >= cutoff).astype(int).to_numpy()
    cutoff_results[cutoff] = run_nested_cv(X, y, make_pipelines(), n_outer_splits=5, n_inner_splits=5, verbose=False)


#============================================================

8. FIRST-PERSON SUBJECT ANALYSIS (spaCy)

#============================================================

In [ ]:
nlp = spacy.load("de_core_news_lg") #using spacy
FIRST_PERSON_FORMS = {
    "ich", "mich", "mir", "wir", "uns",
    "mein", "meine", "meiner", "meinem", "meinen", "meines",
    "unser", "unsere", "unserer", "unserem", "unseren", "unseres",
}#using this list of personal pronouns

def extract_first_person_features(text): #searching the trauma texts (data for this cannot be shared due to privacy concerns)
    doc = nlp(text)
    n_tokens = len([t for t in doc if not t.is_punct and not t.is_space])
    fp_subject_count = sum(
        1 for t in doc if t.text.lower() in FIRST_PERSON_FORMS and t.dep_ in ("sb", "nsubj")
    )
    return {"n_tokens": n_tokens, "fp_subject_count": fp_subject_count,
            "fp_subject_rate": fp_subject_count / n_tokens if n_tokens > 0 else np.nan}

for i in range(len(dataset)):
    text = str(dataset.at[i, "response_trauma"]).strip()
    feats = extract_first_person_features(text) if text else {"n_tokens": np.nan, "fp_subject_count": np.nan, "fp_subject_rate": np.nan}
    for k, v in feats.items():
        dataset.loc[i, f"response_trauma_{k}"] = v

high_group = dataset.loc[dataset[target_col] == positive_class, "response_trauma_fp_subject_rate"].dropna()
low_group = dataset.loc[dataset[target_col] != positive_class, "response_trauma_fp_subject_rate"].dropna()
fp_t, fp_p = ttest_ind(high_group, low_group, equal_var=False)
print(f"First-person subject rate, high vs low PCL-5: t={fp_t:.2f}, p={fp_p:.3f}")


#============================================================

9. PERMUTATION TESTS (slow — run last, budget time accordingly)

#============================================================

In [ ]:
perm_results = {}

# -- narrative conditions + n-gram --
for label, results, X, y in [
    ("Trauma", condition_results["response_trauma"], complete_dataset[embedding_cols_by_condition["response_trauma"]].to_numpy(), y_data),
    ("Negative", condition_results["response_negnt"], complete_dataset[embedding_cols_by_condition["response_negnt"]].to_numpy(), y_data),
    ("Neutral", condition_results["response_neutr"], complete_dataset[embedding_cols_by_condition["response_neutr"]].to_numpy(), y_data),
    ("N-gram", ngram_results, X_text_trauma, y_data),
]:
    fam, _ = pick_best_family(results)
    fam_pipelines = [p for p in (make_pipelines_ngram() if label == "N-gram" else pipelines_list) if p[0] == fam]
    obs, perm_scores, pval = permutation_test_nested_cv(X, y, fam_pipelines, n_permutations=500)
    perm_results[label] = {"observed": obs, "pvalue": pval, "family": fam}

# -- DASS-21 subscales --
for subscale, (tcol, pclass) in dass_targets.items():
    sub_ds = dataset.dropna(subset=embedding_cols_trauma + [tcol]).reset_index(drop=True)
    X = sub_ds[embedding_cols_trauma].to_numpy()
    y = ((sub_ds[tcol] == pclass) * 1).to_numpy()
    fam, _ = pick_best_family(dass_results[subscale])
    fam_pipelines = [p for p in make_pipelines() if p[0] == fam]
    obs, perm_scores, pval = permutation_test_nested_cv(X, y, fam_pipelines, n_permutations=500)
    perm_results[f"DASS {subscale}"] = {"observed": obs, "pvalue": pval, "family": fam}

# -- DSM-5 symptom clusters --
for cluster, (tcol, pclass) in cluster_targets.items():
    sub_ds = dataset.dropna(subset=embedding_cols_trauma + [tcol]).reset_index(drop=True)
    X = sub_ds[embedding_cols_trauma].to_numpy()
    y = ((sub_ds[tcol] == pclass) * 1).to_numpy()
    fam, _ = pick_best_family(cluster_results[cluster])
    fam_pipelines = [p for p in make_pipelines() if p[0] == fam]
    obs, perm_scores, pval = permutation_test_nested_cv(X, y, fam_pipelines, n_permutations=500)
    perm_results[f"Cluster {cluster}"] = {"observed": obs, "pvalue": pval, "family": fam}

#============================================================

10. FINAL METRICS: pooled AUC, participant-level CI, mean-of-folds AUC, precision/recall/F1/accuracy

#============================================================

In [10]:
all_analyses = {
    "Trauma": condition_results["response_trauma"],
    "Negative": condition_results["response_negnt"],
    "Neutral": condition_results["response_neutr"],
    "N-gram baseline": ngram_results,
    "Subjective measures": subjective_results,
    **{f"DASS {k}": v for k, v in dass_results.items()},
    **{f"Cluster {k}": v for k, v in cluster_results.items()},
}

final_metrics = {}
for label, results in all_analyses.items():
    fam, _ = pick_best_family(results)
    pooled = results["per_family_pooled"][fam]
    pooled_auc, ci_low, ci_high = participant_level_single_ci(pooled["y_true"], pooled["y_score"])
    mean_fold_auc = results["per_family_auc"][fam].mean()
    threshold = 0.0 if fam == "svm" else 0.5
    precision, recall, f1, acc = report_precision_recall(pooled, threshold=threshold, label=label)
    final_metrics[label] = {
        "family": fam, "pooled_auc": pooled_auc, "ci": (ci_low, ci_high),
        "mean_fold_auc": mean_fold_auc,
        "precision": precision, "recall": recall, "f1": f1, "accuracy": acc,
    }


NameError: name 'condition_results' is not defined

#============================================================

11. SAVE — ONE FILE, EVERYTHING

#============================================================

In [ ]:
with open("full_results_final.pkl", "wb") as f:
    pickle.dump({
        "final_metrics": final_metrics,
        "perm_results": perm_results,
        "pairwise_comparisons": pairwise_comparisons,
        "correlations_spearman": correlations_spearman,
        "cutoff_results_summary": {
            c: participant_level_single_ci(
                cutoff_results[c]["per_family_pooled"][pick_best_family(cutoff_results[c])[0]]["y_true"],
                cutoff_results[c]["per_family_pooled"][pick_best_family(cutoff_results[c])[0]]["y_score"],
            ) for c in cutoff_results
        },
        "first_person_test": (fp_t, fp_p),
    }, f)

print("Saved full_results_final.pkl")